In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
import re
import string

In [ ]:
data_fake=pd.read_csv('Dataset/Fake.csv')

In [ ]:
data_fake

In [ ]:
data_true=pd.read_csv('Dataset/True.csv')
data_true

In [ ]:
data_fake.head()

In [ ]:
data_fake['class']=0
data_true['class']=1

In [ ]:
data_true.shape,data_fake.shape

In [ ]:
data_fake_manual_testing=data_fake.tail(10)
for i in range (23480,23470,-1):
    data_fake.drop(data_fake.tail(1).index, axis=0, inplace=True)

data_true_manual_testing=data_true.tail(10)
for i in range (21416,21406,-1):
    data_true.drop(data_true.tail(1).index, axis=0, inplace=True)

In [ ]:
data_true.shape,data_fake.shape

In [ ]:
data_fake_manual_testing['class']=0
data_true_manual_testing['class']=1

In [ ]:
data_fake_manual_testing.head(10)

In [ ]:
data_merge=pd.concat([data_fake,data_true],axis=0)
data_merge.head(10)

In [ ]:
data_merge.columns

In [ ]:
data=data_merge.drop(['title','subject','date'],axis=1)

In [ ]:
data.isnull().sum()

In [ ]:
data=data.sample(frac=1)

In [ ]:
data.head()

In [ ]:
data.reset_index(inplace=True)
data.drop(['index'],axis=1,inplace=True)

In [ ]:
data.columns

In [ ]:
data.head()

In [ ]:
def wordopt(text):
    text=text.lower()
    text=re.sub(r'\[.*?]', '',text)
    text=re.sub(r"\\W"," ",text)
    text=re.sub(r'https?://\S+|www\.\S+', '',text)
    text=re.sub(r'<.*?>+', '', text)
    text=re.sub(r'[%s]' % re.escape(string.punctuation), '', text) 
    text=re.sub(r'\n', ' ', text) 
    text=re.sub(r'\w*\d\w*', ' ', text)
    return text

In [ ]:
data['text']=data['text'].apply(wordopt)

In [ ]:
x=data['text']
y=data['class']

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.25)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorization =TfidfVectorizer()
xv_train=vectorization .fit_transform(x_train)
xv_test=vectorization .transform(x_test)

In [ ]:
# Model - logistic Regression
from sklearn.linear_model import LogisticRegression
LR=LogisticRegression()
LR.fit(xv_train,y_train)


In [ ]:
lr_pred=LR.predict(xv_test)

In [ ]:
LR.score(xv_test,y_test)

In [ ]:
print(classification_report(y_test,lr_pred))

In [ ]:
# Model - decision tree
from sklearn.tree import DecisionTreeClassifier
DT=DecisionTreeClassifier()
DT.fit(xv_train,y_train)

In [ ]:
dt_pred=DT.predict(xv_test)

In [ ]:
DT.score(xv_test,y_test)

In [ ]:
print(classification_report(y_test,dt_pred))

In [ ]:
#Model 3 Gradient Boost
from sklearn.ensemble import GradientBoostingClassifier
GB=GradientBoostingClassifier(random_state=0)
GB.fit(xv_train,y_train)

In [ ]:
gb_pred=GB.predict(xv_test)

In [ ]:
GB.score(xv_test,y_test)

In [ ]:
print(classification_report(y_test,gb_pred))

In [ ]:
# Model Rndom forest
from sklearn.ensemble import RandomForestClassifier
RF=RandomForestClassifier(random_state=0)
RF.fit(xv_train,y_train)

In [ ]:
rf_pred=RF.predict(xv_test)

In [ ]:
RF.score(xv_test,y_test)

In [ ]:
print(classification_report(y_test,rf_pred))

In [ ]:
def output_label(n):
    if n == 0:
        return "Fake news"
    elif n == 1:
        return "Not a Fake news"

def manual_testing(news):
    testing_news = {"text": [news]}
    new_def_test = pd.DataFrame(testing_news)

    new_def_test["text"] = new_def_test["text"].apply(wordopt)

    new_x_test = new_def_test["text"]
    new_xv_test = vectorization .transform(new_x_test)

    pred_LR = LR.predict(new_xv_test)
    pred_DT = DT.predict(new_xv_test)
    pred_GB = GB.predict(new_xv_test)
    pred_RF = RF.predict(new_xv_test)

    print(
        "\nLR Prediction : {}".format(output_label(pred_LR[0])))
    print(
        "DT Prediction : {}".format(output_label(pred_DT[0])))
    print(
        "GB Prediction : {}".format(output_label(pred_GB[0])))
    print(
        "RF Prediction : {}".format(output_label(pred_RF[0])))

In [ ]:
news=str(input())
manual_testing(news)

In [ ]:
import pickle

with open("logistic_model.pkl", "wb") as f:
    pickle.dump(LR, f)

with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorization , f)